# Notebook 01 — Exploratory Data Analysis (EDA)

**Purpose:** Explore the raw benchmark data loaded by `01_load_validate.py` and
the per-run statistics computed by `02_compute_stats.py`.
Identify outliers, distributions, and any data quality issues before paper analysis.

**Prerequisites:**
```
python src/analysis/scripts/01_load_validate.py
python src/analysis/scripts/02_compute_stats.py
```

**Inputs:**
- `data/processed/all_runs.parquet` — all raw rows
- `data/processed/stats_paper_s{1,2,3}.parquet` — per-run stats

**Expected runtime:** < 30 s

**Paper scenario mapping:**
- S1 — telemetry 4 msg/s (NTP-dependent `latency_ms`)
- S2 — telemetry 16 msg/s (NTP-dependent `latency_ms`, CSE congestion visible)
- S3 — command ping (primary metric: `cin_create_ms`, NTP-free)

**Authors:** João Parreira, Pedro Barbeiro

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Paths relative to repo root (notebook runs from src/analysis/notebooks/)
REPO_ROOT = Path().resolve().parents[2]
DATA_PROCESSED = REPO_ROOT / 'data' / 'processed'

# Consistent colour palette per protocol
PROTO_COLORS = {
    'mqtt':      '#2196F3',  # blue
    'websocket': '#4CAF50',  # green
    'http':      '#FF9800',  # orange
    'coap':      '#9C27B0',  # purple
}

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print(f'Repo root: {REPO_ROOT}')

## 1. Load data

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / 'all_runs.parquet')

s1 = pd.read_parquet(DATA_PROCESSED / 'stats_paper_s1.parquet')
s2 = pd.read_parquet(DATA_PROCESSED / 'stats_paper_s2.parquet')
s3 = pd.read_parquet(DATA_PROCESSED / 'stats_paper_s3.parquet')

print(f'all_runs: {len(df):,} rows')
print(f'  protocols: {sorted(df.protocol.unique())}')
print(f'  paper_scenarios: {sorted(df.paper_scenario.unique())}')
print(f'S1 runs: {len(s1)}, S2 runs: {len(s2)}, S3 runs: {len(s3)}')

## 2. Coverage — runs per protocol × scenario

In [ ]:
coverage = df.groupby(['protocol', 'paper_scenario'])['run_id'].nunique().unstack(fill_value=0)
coverage.columns = [f'S{c}' for c in coverage.columns]
print('Runs collected per (protocol, paper scenario):')
print(coverage.to_string())
print('\nTarget: 10 runs per cell (4 protocols × 3 scenarios = 120 total)')

## 3. S1 — Telemetry 4 msg/s: latency distribution

In [ ]:
df_s1 = df[df['paper_scenario'] == 1]
protocols_s1 = sorted(df_s1['protocol'].unique())

fig, axes = plt.subplots(1, len(protocols_s1), figsize=(5 * len(protocols_s1), 4), sharey=True)
if len(protocols_s1) == 1:
    axes = [axes]

for ax, proto in zip(axes, protocols_s1):
    data = df_s1[df_s1['protocol'] == proto]['latency_ms'].dropna()
    ax.hist(data, bins=40, color=PROTO_COLORS.get(proto, 'steelblue'), alpha=0.8, edgecolor='white')
    ax.set_title(proto)
    ax.set_xlabel('latency_ms')
    ax.set_ylabel('count')
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.2, label=f'median={data.median():.0f}')
    ax.legend(fontsize=9)

fig.suptitle('S1 (4 msg/s) — latency_ms distribution (NTP-inflated)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. S1 — Per-run mean latency (run-to-run stability)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for proto in sorted(s1['protocol'].unique()):
    grp = s1[s1['protocol'] == proto].sort_values('run_id')
    ax.plot(range(len(grp)), grp['lat_mean'], marker='o',
            label=proto, color=PROTO_COLORS.get(proto, None))
ax.set_xlabel('run index')
ax.set_ylabel('lat_mean (ms)')
ax.set_title('S1 — per-run mean latency (run-to-run variability)')
ax.legend()
plt.tight_layout()
plt.show()

print(s1.groupby('protocol')[['lat_mean', 'lat_std', 'packet_loss_frac', 'tput_msg_per_s']].describe())

## 5. S2 — Telemetry 16 msg/s: CSE congestion visible

In [ ]:
df_s2 = df[df['paper_scenario'] == 2]

if df_s2.empty:
    print('[INFO] No S2 data yet — run WS/MQTT/HTTP/CoAP S2 benchmarks')
else:
    protocols_s2 = sorted(df_s2['protocol'].unique())
    fig, axes = plt.subplots(1, len(protocols_s2), figsize=(5 * len(protocols_s2), 4), sharey=True)
    if len(protocols_s2) == 1:
        axes = [axes]
    for ax, proto in zip(axes, protocols_s2):
        # Plot latency_ms vs seq to show queue growth over the run
        data = df_s2[df_s2['protocol'] == proto].sort_values('seq')
        ax.scatter(data['seq'], data['latency_ms'], s=4,
                   color=PROTO_COLORS.get(proto, 'steelblue'), alpha=0.5)
        ax.set_title(proto)
        ax.set_xlabel('seq')
        ax.set_ylabel('latency_ms')
    fig.suptitle('S2 (16 msg/s) — latency_ms vs seq (CSE queue growth)', fontsize=13)
    plt.tight_layout()
    plt.show()

    print(s2.groupby('protocol')[['lat_mean', 'lat_p95', 'packet_loss_frac']].describe())

## 6. S3 — Command ping: cin_create_ms distribution

In [ ]:
df_s3 = df[df['paper_scenario'] == 3]
protocols_s3 = sorted(df_s3['protocol'].unique())

fig, axes = plt.subplots(1, len(protocols_s3), figsize=(5 * len(protocols_s3), 4), sharey=True)
if len(protocols_s3) == 1:
    axes = [axes]

for ax, proto in zip(axes, protocols_s3):
    data = df_s3[df_s3['protocol'] == proto]['cin_create_ms'].dropna()
    ax.hist(data, bins=30, color=PROTO_COLORS.get(proto, 'steelblue'), alpha=0.8, edgecolor='white')
    ax.set_title(proto)
    ax.set_xlabel('cin_create_ms')
    ax.set_ylabel('count')
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.2,
               label=f'median={data.median():.0f} ms')
    ax.legend(fontsize=9)

fig.suptitle('S3 (command ping) — cin_create_ms distribution (NTP-free)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. S3 — Packet loss per run

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for proto in sorted(s3['protocol'].unique()):
    grp = s3[s3['protocol'] == proto].sort_values('run_id')
    ax.plot(range(len(grp)), grp['packet_loss_frac'] * 100, marker='o',
            label=proto, color=PROTO_COLORS.get(proto, None))
ax.set_xlabel('run index')
ax.set_ylabel('packet loss (%)')
ax.set_title('S3 — packet loss per run (ACK timeout-based)')
ax.legend()
plt.tight_layout()
plt.show()

print(s3.groupby('protocol')[['cin_mean', 'cin_std', 'cin_p95', 'packet_loss_frac']].describe())

## 8. Protocol overhead comparison (all scenarios)

In [ ]:
all_stats = pd.concat([s1.assign(scenario='S1'), s2.assign(scenario='S2'), s3.assign(scenario='S3')])

fig, ax = plt.subplots(figsize=(8, 4))
pivot = all_stats.groupby(['protocol', 'scenario'])['overhead_pct_mean'].mean().unstack('scenario')
pivot.plot(kind='bar', ax=ax, color=[PROTO_COLORS.get(p, 'gray') for p in pivot.index])
ax.set_xlabel('protocol')
ax.set_ylabel('overhead (%)')
ax.set_title('Protocol overhead (header / (header + payload) × 100) by scenario')
ax.legend(title='scenario')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()